In [ ]:
import re
import numpy as np


from collections import Counter

# -----------------------------
# 1. Dataset
# -----------------------------
sentence_pairs = [
    ("i am happy", "मैं खुश हूँ"),
    ("i am sad", "मैं दुखी हूँ"),
    ("i am a student", "मैं एक छात्र हूँ"),
    ("you are my friend", "आप मेरे दोस्त हैं"),
    ("he is a teacher", "वह एक शिक्षक है"),
    ("she is reading a book", "वह एक किताब पढ़ रही है"),
    ("i like tea", "मुझे चाय पसंद है"),
    ("i like coffee", "मुझे कॉफी पसंद है"),
    ("we are going home", "हम घर जा रहे हैं"),
    ("they are playing", "वे खेल रहे हैं"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is rahul", "मेरा नाम राहुल है"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("good morning", "सुप्रभात"),
    ("good night", "शुभ रात्रि"),
    ("thank you", "धन्यवाद"),
    ("see you tomorrow", "कल मिलते हैं"),
    ("where are you going", "आप कहाँ जा रहे हैं"),
    ("i live in bangalore", "मैं बैंगलोर में रहता हूँ"),
    ("this is my book", "यह मेरी किताब है"),
    ("that is a car", "वह एक कार है"),
    ("we love india", "हम भारत से प्यार करते हैं"),
    ("open the door", "दरवाजा खोलो"),
    ("close the window", "खिड़की बंद करो"),
    ("the child is sleeping", "बच्चा सो रहा है"),
    ("the sun is bright", "सूरज तेज है"),
    ("today is a holiday", "आज छुट्टी है"),
    ("i am learning ai", "मैं एआई सीख रहा हूँ"),
    ("this is a simple example", "यह एक सरल उदाहरण है")
]

# -----------------------------
# 2. Text cleaning functions
# -----------------------------
def clean_english_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def clean_hindi_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

# -----------------------------
# 3. Clean dataset
# -----------------------------
cleaned_pairs = []
for eng, hin in sentence_pairs:
    eng_clean = clean_english_text(eng)
    hin_clean = clean_hindi_text(hin)
    cleaned_pairs.append((eng_clean, hin_clean))

# -----------------------------
# 4. Add special tokens to target sentences
# -----------------------------
START_TOKEN = "<start>"
END_TOKEN = "<end>"
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

processed_pairs = []
for eng, hin in cleaned_pairs:
    hin_with_tokens = f"{START_TOKEN} {hin} {END_TOKEN}"
    processed_pairs.append((eng, hin_with_tokens))

# -----------------------------
# 5. Tokenization (simple split-based)
# -----------------------------
def tokenize(text):
    return text.split()

source_sentences = [eng for eng, hin in processed_pairs]
target_sentences = [hin for eng, hin in processed_pairs]

source_tokenized = [tokenize(sent) for sent in source_sentences]
target_tokenized = [tokenize(sent) for sent in target_sentences]

# -----------------------------
# 6. Build vocabularies
# -----------------------------
def build_vocab(tokenized_sentences, add_special_tokens=True):
    counter = Counter()
    for sent in tokenized_sentences:
        counter.update(sent)

    vocab = {}
    index = 0

    if add_special_tokens:
        vocab[PAD_TOKEN] = index
        index += 1
        vocab[UNK_TOKEN] = index
        index += 1

    for word, _ in counter.items():
        if word not in vocab:
            vocab[word] = index
            index += 1

    return vocab

source_vocab = build_vocab(source_tokenized, add_special_tokens=True)
target_vocab = build_vocab(target_tokenized, add_special_tokens=True)

# Reverse vocabularies for decoding
source_index_to_word = {idx: word for word, idx in source_vocab.items()}
target_index_to_word = {idx: word for word, idx in target_vocab.items()}

# -----------------------------
# 7. Convert tokenized sentences to integer sequences
# -----------------------------
def sentence_to_sequence(tokenized_sentence, vocab):
    return [vocab.get(word, vocab[UNK_TOKEN]) for word in tokenized_sentence]

source_sequences = [sentence_to_sequence(sent, source_vocab) for sent in source_tokenized]
target_sequences = [sentence_to_sequence(sent, target_vocab) for sent in target_tokenized]

# -----------------------------
# 8. Find maximum lengths
# -----------------------------
max_source_len = max(len(seq) for seq in source_sequences)
max_target_len = max(len(seq) for seq in target_sequences)

# -----------------------------
# 9. Pad sequences
# -----------------------------
def pad_sequence(sequence, max_len, pad_value=0):
    return sequence + [pad_value] * (max_len - len(sequence))

encoder_input_data = np.array([
    pad_sequence(seq, max_source_len, pad_value=source_vocab[PAD_TOKEN])
    for seq in source_sequences
])

target_padded = np.array([
    pad_sequence(seq, max_target_len, pad_value=target_vocab[PAD_TOKEN])
    for seq in target_sequences
])

# -----------------------------
# 10. Prepare decoder input and decoder target
# -----------------------------
decoder_input_data = target_padded[:, :-1]
decoder_target_data = target_padded[:, 1:]

# Expand last dimension for sparse categorical crossentropy later
decoder_target_data = np.expand_dims(decoder_target_data, -1)

# -----------------------------
# 11. Display summary
# -----------------------------
print("Total sentence pairs:", len(processed_pairs))
print("Source vocabulary size:", len(source_vocab))
print("Target vocabulary size:", len(target_vocab))
print("Max source sentence length:", max_source_len)
print("Max target sentence length:", max_target_len)

print("\nSample processed pair:")
print("English:", processed_pairs[0][0])
print("Hindi :", processed_pairs[0][1])

print("\nSample source tokenized:", source_tokenized[0])
print("Sample target tokenized:", target_tokenized[0])

print("\nSample source sequence:", source_sequences[0])
print("Sample target sequence:", target_sequences[0])

print("\nEncoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

# -----------------------------
# 12. Show one example clearly
# -----------------------------
example_index = 0
print("\n--- Example walkthrough ---")
print("Original English sentence :", source_sentences[example_index])
print("Original Hindi sentence   :", target_sentences[example_index])
print("Encoder input IDs         :", encoder_input_data[example_index])
print("Decoder input IDs         :", decoder_input_data[example_index])
print("Decoder target IDs        :", decoder_target_data[example_index].flatten())

print("\nDecoder input words:")
print([target_index_to_word.get(idx, UNK_TOKEN) for idx in decoder_input_data[example_index]])

print("Decoder target words:")
print([target_index_to_word.get(idx, UNK_TOKEN) for idx in decoder_target_data[example_index].flatten()])


#-------------------------------------------------------

from tensorflow.keras.layers import Embedding
import tensorflow as tf

from tensorflow.keras.layers import Embedding
import tensorflow as tf

sample_vocab_size = len(source_vocab)
embedding_dim = 8

embedding_layer = Embedding(input_dim=sample_vocab_size, output_dim=embedding_dim)

sample_input = tf.constant([[2, 3, 4]])
sample_output = embedding_layer(sample_input)

print("Input shape :", sample_input.shape)
print("Output shape:", sample_output.shape)
print(sample_output.numpy())


#--------------------------------------
for i in range(5):
    print(processed_pairs[i])


#---------------------------------------

print(source_tokenized[0])
print(target_tokenized[0])

#------------------------------------


print("Source vocab sample:", list(source_vocab.items())[:10])
print("Target vocab sample:", list(target_vocab.items())[:10])



#----------------------------------------

print("English sentence :", source_sentences[0])
print("Sequence         :", source_sequences[0])

print("Hindi sentence   :", target_sentences[0])
print("Sequence         :", target_sequences[0])


#-----------------------------------------

print("Before padding:", source_sequences[0])
print("After padding :", encoder_input_data[0])


#------------------------------------------
print("Decoder input IDs :", decoder_input_data[0])
print("Decoder target IDs:", decoder_target_data[0].flatten())

print("Decoder input words :",
      [target_index_to_word[idx] for idx in decoder_input_data[0]])

print("Decoder target words:",
      [target_index_to_word[idx] for idx in decoder_target_data[0].flatten()])